# SteppeDNA: GNN on AlphaFold 3D Protein Structure

This notebook builds a Graph Neural Network (GNN) that operates on AlphaFold-predicted
3D protein structures. Each residue is a node, edges connect spatially close residues
(within 8 angstroms), and the GNN learns structural features for pathogenicity prediction.

**Requirements:** Google Colab with GPU

**What this does:**
1. Downloads AlphaFold structures for 5 HR genes
2. Builds residue-level contact graphs
3. Trains a GNN (GCN/GAT) for structural feature extraction
4. Exports per-variant structural embeddings
5. These become additional features in the SteppeDNA ensemble

In [ ]:
!pip install -q torch torch-geometric biopython pandas numpy scikit-learn requests

In [ ]:
from google.colab import files
print("Upload master_training_dataset.csv:")
uploaded = files.upload()

In [ ]:
import torch
import numpy as np
import pandas as pd
import requests
import io
from collections import defaultdict

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Step 1: Download AlphaFold structures
UNIPROT_IDS = {
    'BRCA1': 'P38398', 'BRCA2': 'P51587', 'PALB2': 'Q86YC2',
    'RAD51C': 'O43502', 'RAD51D': 'O75771',
}
GENE_AA_LENGTHS = {'BRCA1': 1863, 'BRCA2': 3418, 'PALB2': 1186, 'RAD51C': 376, 'RAD51D': 328}

structures = {}

for gene, uid in UNIPROT_IDS.items():
    # AlphaFold DB API
    url = f'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb'
    resp = requests.get(url)
    if resp.ok:
        structures[gene] = resp.text
        n_lines = len([l for l in resp.text.split('\n') if l.startswith('ATOM')])
        print(f'{gene}: Downloaded ({n_lines} ATOM records)')
    else:
        print(f'{gene}: FAILED ({resp.status_code})')
        # For large proteins (BRCA2 > 2700aa), AlphaFold may have fragments
        # Try fragment 1
        url2 = f'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb'
        resp2 = requests.get(url2)
        if resp2.ok:
            structures[gene] = resp2.text
            print(f'  -> Got fragment 1')

print(f'\nStructures loaded: {len(structures)}/5')

In [ ]:
# Step 2: Parse PDB into residue-level features and contact graphs

def parse_pdb_to_graph(pdb_text, distance_threshold=8.0):
    """Parse PDB text into a residue graph.
    
    Nodes: residues with features [x, y, z, bfactor, is_helix, is_sheet]
    Edges: residues within distance_threshold angstroms (C-alpha distance)
    """
    residues = {}  # resid -> {coord, bfactor, ...}
    
    for line in pdb_text.split('\n'):
        if not line.startswith('ATOM'):
            continue
        atom_name = line[12:16].strip()
        if atom_name != 'CA':  # Only C-alpha atoms
            continue
        
        resid = int(line[22:26].strip())
        x = float(line[30:38].strip())
        y = float(line[38:46].strip())
        z = float(line[46:54].strip())
        bfactor = float(line[60:66].strip())
        resname = line[17:20].strip()
        
        residues[resid] = {
            'coord': np.array([x, y, z]),
            'bfactor': bfactor,
            'resname': resname,
        }
    
    # Sort by residue ID
    sorted_resids = sorted(residues.keys())
    n_residues = len(sorted_resids)
    resid_to_idx = {r: i for i, r in enumerate(sorted_resids)}
    
    # Node features: [x, y, z, bfactor_normalized]
    coords = np.array([residues[r]['coord'] for r in sorted_resids])
    bfactors = np.array([residues[r]['bfactor'] for r in sorted_resids])
    
    # Normalize
    coords_norm = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
    bf_norm = (bfactors - bfactors.mean()) / (bfactors.std() + 1e-8)
    
    node_features = np.column_stack([coords_norm, bf_norm[:, None]])
    
    # Build edges: connect residues within distance threshold
    edges_src = []
    edges_dst = []
    
    from scipy.spatial.distance import cdist
    dist_matrix = cdist(coords, coords)
    
    for i in range(n_residues):
        for j in range(i + 1, n_residues):
            if dist_matrix[i, j] < distance_threshold:
                edges_src.extend([i, j])
                edges_dst.extend([j, i])
    
    edge_index = np.array([edges_src, edges_dst], dtype=np.int64)
    
    return {
        'node_features': node_features,
        'edge_index': edge_index,
        'coords': coords,
        'resid_to_idx': resid_to_idx,
        'sorted_resids': sorted_resids,
        'n_residues': n_residues,
        'n_edges': len(edges_src),
    }

graphs = {}
for gene, pdb_text in structures.items():
    g = parse_pdb_to_graph(pdb_text)
    graphs[gene] = g
    print(f'{gene}: {g["n_residues"]} residues, {g["n_edges"]} edges')

In [ ]:
# Step 3: Define GNN model
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch_geometric.data import Data, Batch

class StructureGNN(nn.Module):
    """GNN that learns structural representations from protein contact graphs."""
    def __init__(self, in_channels=4, hidden_channels=64, out_channels=32, n_layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(n_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.convs.append(GCNConv(hidden_channels, out_channels))
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = self.dropout(x)
        return x  # Per-node embeddings

gnn = StructureGNN(in_channels=4, hidden_channels=64, out_channels=32).to(DEVICE)
print(f'GNN parameters: {sum(p.numel() for p in gnn.parameters()):,}')

In [ ]:
# Step 4: Train GNN with self-supervised objective
# We use a denoising objective: mask node features and predict them

optimizer = torch.optim.Adam(gnn.parameters(), lr=1e-3, weight_decay=1e-5)
reconstruction_head = nn.Linear(32, 4).to(DEVICE)  # Predict original node features
opt_head = torch.optim.Adam(reconstruction_head.parameters(), lr=1e-3)

EPOCHS = 100
MASK_RATE = 0.15

gnn.train()
reconstruction_head.train()

for epoch in range(EPOCHS):
    total_loss = 0
    n_graphs = 0
    
    for gene, g in graphs.items():
        x = torch.tensor(g['node_features'], dtype=torch.float32).to(DEVICE)
        edge_index = torch.tensor(g['edge_index'], dtype=torch.long).to(DEVICE)
        
        # Mask random nodes
        mask = torch.rand(x.shape[0]) < MASK_RATE
        x_masked = x.clone()
        x_masked[mask] = 0  # Zero out masked nodes
        
        optimizer.zero_grad()
        opt_head.zero_grad()
        
        node_embeds = gnn(x_masked, edge_index)
        reconstructed = reconstruction_head(node_embeds)
        
        # Loss only on masked nodes
        loss = F.mse_loss(reconstructed[mask], x[mask])
        loss.backward()
        
        optimizer.step()
        opt_head.step()
        
        total_loss += loss.item()
        n_graphs += 1
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/n_graphs:.6f}')

print('\nGNN training complete!')

In [ ]:
# Step 5: Extract per-variant structural embeddings
import pickle

df = pd.read_csv('master_training_dataset.csv')
print(f'Dataset: {len(df)} variants')

# Detect the gene column name
gene_col = None
for col in ['gene_name', 'gene', 'Gene', 'Gene_Name']:
    if col in df.columns:
        gene_col = col
        break

if gene_col:
    print(f'Gene column: "{gene_col}"')
    print(f'Gene distribution:\n{df[gene_col].value_counts().to_string()}')
else:
    print('WARNING: No gene column found! Will treat all as BRCA2.')

gnn.eval()

# Pre-compute all node embeddings for each gene
gene_node_embeds = {}
for gene, g in graphs.items():
    x = torch.tensor(g['node_features'], dtype=torch.float32).to(DEVICE)
    edge_index = torch.tensor(g['edge_index'], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        node_embeds = gnn(x, edge_index)
    gene_node_embeds[gene] = node_embeds.cpu().numpy()
    print(f'{gene}: {node_embeds.shape} node embeddings')

# Map each variant to its residue embedding
all_embeddings = {}
NEIGHBOR_RADIUS = 5

for idx, row in df.iterrows():
    gene = row[gene_col] if gene_col else 'BRCA2'
    if gene not in graphs:
        continue
    
    g = graphs[gene]
    embeds = gene_node_embeds[gene]
    
    # Try multiple column names for AA position
    aa_pos = None
    if 'relative_aa_pos' in row.index and pd.notna(row['relative_aa_pos']):
        aa_pos = int(row['relative_aa_pos'] * GENE_AA_LENGTHS.get(gene, 3418))
    elif 'AA_pos' in row.index and pd.notna(row['AA_pos']):
        aa_pos = int(row['AA_pos'])
    elif 'cDNA_pos' in row.index and pd.notna(row['cDNA_pos']):
        aa_pos = int(row['cDNA_pos']) // 3
    else:
        aa_pos = len(g['sorted_resids']) // 2
    
    aa_pos = max(1, aa_pos)
    
    # Find closest residue in structure
    if aa_pos in g['resid_to_idx']:
        center_idx = g['resid_to_idx'][aa_pos]
    else:
        dists = [abs(r - aa_pos) for r in g['sorted_resids']]
        center_idx = np.argmin(dists)
    
    # Average over neighborhood
    start = max(0, center_idx - NEIGHBOR_RADIUS)
    end = min(len(embeds), center_idx + NEIGHBOR_RADIUS + 1)
    variant_embed = embeds[start:end].mean(axis=0)
    
    all_embeddings[idx] = variant_embed

print(f'\nGenerated {len(all_embeddings)} structural embeddings (dim={embeds.shape[1]})')

In [ ]:
# Step 6: Save
output = {
    'embeddings': all_embeddings,
    'embedding_dim': 32,
    'gnn_config': {
        'in_channels': 4,
        'hidden_channels': 64,
        'out_channels': 32,
        'n_layers': 3,
        'distance_threshold': 8.0,
        'training_epochs': EPOCHS,
    },
    'gene_coverage': list(graphs.keys()),
}

with open('gnn_structural_embeddings.pkl', 'wb') as f:
    pickle.dump(output, f)

# Save GNN weights
torch.save(gnn.state_dict(), 'gnn_model_weights.pt')

print('Saved gnn_structural_embeddings.pkl and gnn_model_weights.pt')

In [ ]:
# Download
from google.colab import files
files.download('gnn_structural_embeddings.pkl')
files.download('gnn_model_weights.pt')

print('\n--- DONE ---')
print('Place gnn_structural_embeddings.pkl in SteppeDNA/data/')
print('These 32 GNN features will be added to the existing 103 features')
print('Update feature_engineering.py to incorporate GNN embeddings')
print('Retrain with: python scripts/train_universal_model.py')